# Phase 1 — Raw Data Profiling

Goal: understand the raw `placements.json` completely before any cleaning or modeling.  
This notebook is the proof that the data was approached analytically, not just pipelined.

**Dataset:** `data/raw/placements.json`  
**Profiler:** `src/ingestion/load_json.profile_placements()`  
**Output:** observations → `docs/assumptions.md`

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Allow imports from project root
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion.load_json import load_placements, profile_placements

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Load and profile

In [ ]:
records = load_placements(ROOT / "data" / "raw" / "placements.json")
profile = profile_placements(records)

vol = profile["volume"]
print(f"Records : {vol['record_count']}")
print(f"Offers  : {vol['offer_count']}")
print(f"Avg offers/record: {vol['offers_per_record']['mean']}")
print(f"Max offers/record: {vol['offers_per_record']['max']}")

## 2. Schema snapshot

In [ ]:
schema = profile["schema"]
print("--- Parent-record keys ---")
for k, freq in schema["record_key_frequency"].items():
    print(f"  {k:<25} present in {freq}/{vol['record_count']} records")

print("\n--- Offer keys ---")
for k, freq in schema["offer_key_frequency"].items():
    print(f"  {k:<25} present in {freq}/{vol['offer_count']} offers")

**Note:** `branchesAllowed` and `eligibilityCgpa` appear at the parent-record level in 98 of 461 records — a data-entry artefact. Phase 2 flattening must handle this spillover explicitly.

## 3. Offers-per-record distribution

In [ ]:
dist = vol["offers_per_record"]["distribution"]
x = [int(k) for k in dist]
y = list(dist.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x, y, color="steelblue", edgecolor="white")
ax.set_xlabel("Offers per company record")
ax.set_ylabel("Number of records")
ax.set_title("Offers-per-Record Distribution")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for xi, yi in zip(x, y):
    ax.text(xi, yi + 1, str(yi), ha="center", fontsize=8)
plt.tight_layout()
plt.show()
print(
    "76% of companies appear with exactly one offer. Multi-offer records exist up to 12."
)

## 4. Field-level missingness — offer level

In [ ]:
miss_offer = profile["missingness"]["offer_level"]
miss_df = pd.DataFrame(
    [
        {
            "field": k,
            "missing_pct": v["missing_pct"],
            "missing_count": v["missing_count"],
        }
        for k, v in miss_offer.items()
    ]
).sort_values("missing_pct", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [
    "#d73027" if p > 50 else "#f46d43" if p > 20 else "#74add1"
    for p in miss_df["missing_pct"]
]
bars = ax.barh(miss_df["field"], miss_df["missing_pct"], color=colors)
ax.set_xlabel("Missing %")
ax.set_title("Offer-level Field Missingness")
ax.axvline(20, color="gray", linestyle="--", linewidth=0.8, label="20% threshold")
ax.axvline(50, color="red", linestyle="--", linewidth=0.8, label="50% threshold")
ax.legend(fontsize=8)
for bar, pct in zip(bars, miss_df["missing_pct"]):
    ax.text(
        pct + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}%",
        va="center",
        fontsize=8,
    )
plt.tight_layout()
plt.show()

In [ ]:
print(miss_df.to_string(index=False))

**Key observations:**
- `branchesNote`, `eligibilityNote`, `stipendNote` are all > 75% empty — treat as optional enrichment only.
- `branchwiseBreakup` missing in 57.8% of offers — not a reliable field for analytical logic.
- `ctcNote` is present for 61% of offers and contains structured hints (location, gross salary, work mode).
- Core fields (`ctc`, `stipend`, `branchesAllowed`, `jobRole`) are mostly present but not uniformly clean.

## 5. CTC format analysis

In [ ]:
ctc_patterns = profile["format_variants"]["ctc"]["pattern_counts"]
ctc_examples = profile["format_variants"]["ctc"]["pattern_examples"]

labels = list(ctc_patterns.keys())
sizes = list(ctc_patterns.values())
colors_pie = ["#2196F3", "#FF9800", "#4CAF50", "#F44336", "#9C27B0", "#009688"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

wedges, texts, autotexts = ax1.pie(
    sizes,
    labels=labels,
    autopct="%1.1f%%",
    colors=colors_pie[: len(labels)],
    startangle=140,
    pctdistance=0.8,
)
ax1.set_title("CTC Field — Pattern Distribution (654 offers)")

ax2.barh(labels, sizes, color=colors_pie[: len(labels)], edgecolor="white")
ax2.set_xlabel("Count")
ax2.set_title("CTC Pattern Counts")
for i, (label, size) in enumerate(zip(labels, sizes)):
    ax2.text(size + 1, i, str(size), va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nExamples per pattern:")
for pattern, examples in ctc_examples.items():
    print(f"  {pattern:<15}: {examples}")

**CTC parsing requirements for Phase 3:**
- `INTEGER` (406): raw value is in rupees (e.g. `411000` → 4.11 LPA). Divide by 100,000.
- `RANGE` (135): e.g. `800000-1200000` → min/max/midpoint in LPA.
- `EMPTY` (68): status = `MISSING`.
- `UNKNOWN_TEXT` (12): `Not Disclosed`, `Not Declared` → status = `UNKNOWN`.
- `PENDING_TEXT` (6): `To be notified`, `Negotiable` → status = `PENDING`.
- `OTHER` (27): free text requiring case-by-case handling or status = `UNKNOWN`.

## 6. Stipend format analysis

In [ ]:
stip_patterns = profile["format_variants"]["stipend"]["pattern_counts"]
stip_examples = profile["format_variants"]["stipend"]["pattern_examples"]

labels_s = list(stip_patterns.keys())
sizes_s = list(stip_patterns.values())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    labels_s,
    sizes_s,
    color=["#2196F3", "#FF9800", "#4CAF50", "#F44336", "#9C27B0"][: len(labels_s)],
    edgecolor="white",
)
ax.set_title("Stipend Field — Pattern Distribution (654 offers)")
ax.set_ylabel("Count")
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        str(int(bar.get_height())),
        ha="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print("\nExamples per pattern:")
for pattern, examples in stip_examples.items():
    print(f"  {pattern:<15}: {examples}")

**Stipend parsing requirements for Phase 3:**
- `INTEGER` (403): monthly amount in rupees. Store as-is.
- `EMPTY` (181): 27.7% — status = `MISSING`. Correlates with `hasStipend = false`.
- `RANGE` (48): e.g. `50000-75000` → min/max/midpoint.
- `UNKNOWN_TEXT` (12): status = `UNKNOWN`.
- `OTHER` (10): unicode dashes (–) used instead of hyphens, free text amounts — needs regex fallback.

## 7. Offer type distribution

In [ ]:
type_variants = profile["variants"]["type"]["top_values"]
type_df = pd.DataFrame(type_variants)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(type_df["value"], type_df["count"], color="#5C6BC0", edgecolor="white")
ax.set_title("Offer Type Distribution (654 offers)")
ax.set_xlabel("Count")
for bar, count in zip(bars, type_df["count"]):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{count} ({count/654*100:.1f}%)",
        va="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

**Type taxonomy for Phase 3:**
- `Intern+Performance Based Chance of FTE` (45.4%) → standardize to `INTERN_POSSIBLE_FTE`
- `FTE` (26.0%) → `FTE`
- `Intern+FTE` (15.1%) → `INTERN_FTE`
- `Intern` (9.2%) → `INTERN`
- `PPO (Summer Intern/Competition)` (4.1%) → `PPO`
- `PPO from Summer Intern` (0.2%) → merge into `PPO`

## 8. Branch distribution

In [ ]:
branch_variants = profile["variants"]["branchesAllowed"]["top_values"]
branch_df = pd.DataFrame(branch_variants)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    branch_df["value"], branch_df["count"], color="#26A69A", edgecolor="white"
)
ax.set_title("Top 15 Branch Codes (across all offer-branch pairs)")
ax.set_xlabel("Appearances")
for bar, count in zip(bars, branch_df["count"]):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        str(count),
        va="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

branch_len_dist = profile["variants"]["branchesAllowed"]["list_length_distribution"]
print("Branch-list length distribution per offer:")
for length, count in branch_len_dist.items():
    print(f"  {length} branches → {count} offers")

**Branch findings:**
- 23 unique branch codes total.
- Top 4 (COPC, COE, COBS, ENC) appear in 300+ offers each — likely CS/IT/ECE-family codes.
- `B.E. All Branches` (62 appearances) is a catch-all that maps to `ALL_BRANCHES`.
- Branch lists range from 0 to 14 entries per offer — confirms bridge table is required.
- Only 6 offers have zero branches — handle separately in Phase 2.

## 9. Role variant analysis

In [ ]:
role_variants = profile["variants"]["jobRole"]
print(
    f"Unique raw jobRole values: {role_variants['unique_count']} across {vol['offer_count']} offers"
)
print(
    f"Fragmentation ratio: {role_variants['unique_count'] / vol['offer_count']:.2f} (1.0 = every offer has unique role)"
)
print()
print("Top 15 raw role values:")
top_roles = pd.DataFrame(role_variants["top_values"])
print(top_roles.to_string(index=False))

In [ ]:
top15 = top_roles.head(15)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top15["value"], top15["count"], color="#7E57C2", edgecolor="white")
ax.set_title("Top 15 Raw jobRole Values")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

**Role normalization challenge:**
- 518 unique values across 654 offers = 0.79 fragmentation ratio. Extremely high noise.
- `GET`, `Graduate Engineer Trainee`, `Graduate Engineer Trainees (GETs)` are the same role — require deduplication.
- `Not Known`, `Not Declared`, empty strings (13+ offers) — status = `UNKNOWN`.
- Phase 3 will need a two-level taxonomy: `role_standardized` + `job_family`.
- `rapidfuzz` fuzzy matching will be critical for low-frequency variants.

## 10. Note field signal

In [ ]:
notes = profile["notes"]
note_df = pd.DataFrame(
    [
        {
            "field": k,
            "non_empty_count": v["non_empty_count"],
            "non_empty_pct": v["non_empty_pct"],
        }
        for k, v in notes.items()
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    note_df["field"], note_df["non_empty_pct"], color="#EF5350", edgecolor="white"
)
ax.set_title("Note Fields — Non-Empty Rate (% of 654 offers)")
ax.set_ylabel("% non-empty")
ax.set_ylim(0, 75)
for bar, pct in zip(bars, note_df["non_empty_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{pct:.1f}%",
        ha="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print("\nSample ctcNote values (showing structured hints):")
for ex in notes["ctcNote"]["example_values"]:
    print(f"  - {ex}")

**Note extraction priorities for Phase 3 (`extract_notes.py`):**
- `ctcNote` (61% present): location city, work mode (Hybrid/Remote/Onsite), gross salary mentions.
- `stipendNote` (22.6% present): internship duration in months, monthly vs lump-sum disambiguation.
- `eligibilityNote` (15.7% present): CGPA thresholds, backlog rules.
- `branchesNote` (14.8% present): M.E./M.Tech branch restrictions not in the list field.

Rule-based regex extraction is sufficient and defensible here.

## 11. CGPA / Eligibility variants

In [ ]:
cgpa_variants = profile["variants"]["eligibilityCgpa"]
print(f"Unique eligibilityCgpa values: {cgpa_variants['unique_count']}")
print()
print("All values (sorted by frequency):")
for item in cgpa_variants["top_values"]:
    print(f"  {item['value']!r:<40} → {item['count']}")

**CGPA parsing notes:**
- Most values are explicit thresholds (e.g. `6.5`, `7.0`, `7.5`, `8.0`).
- `No CGPA Criteria` is a meaningful positive signal — should be a flag column `no_cgpa_criteria = True`.
- `Not Known`, `Not Declared`, empty → status = `UNKNOWN`.

## 12. studentsSelected variants

In [ ]:
sel_variants = profile["variants"]["studentsSelected"]
print(f"Unique studentsSelected values: {sel_variants['unique_count']}")
print()
print("Top 15 values:")
for item in sel_variants["top_values"]:
    print(f"  {item['value']!r:<40} → {item['count']}")

**studentsSelected parsing notes:**
- Many values are pending/unknown strings, not numbers.
- Numeric extraction possible for values like `"3"`, `"5"`, `"10"`.
- `Process Pending` is the most common non-numeric value — status = `PENDING`.

## 13. Summary: key findings for Phase 2 and Phase 3

In [ ]:
summary = {
    "Records": vol["record_count"],
    "Offers": vol["offer_count"],
    "CTC parseable (INTEGER+RANGE)": 406 + 135,
    "CTC not parseable": 68 + 6 + 27 + 12,
    "Stipend parseable (INTEGER+RANGE)": 403 + 48,
    "Stipend missing": 181,
    "Unique raw jobRole values": 518,
    "Unique branch codes": 23,
    "Offer types": 6,
    "ctcNote non-empty": 399,
    "Parent-record spillover fields": 98,
}

for k, v in summary.items():
    print(f"  {k:<45} {v}")

## Phase 1 complete

All observations recorded. Detailed written version in `docs/assumptions.md`.

**Next:** Phase 2 — Flattening & Ingestion (`src/ingestion/flatten_offers.py`)